In [12]:
import pandas as pd
from astroquery.gaia import Gaia
import numpy as np

In [23]:
def clean_jobids():

    # Access to the Gaia Archive as registered user
    #Gaia.login()
    # Retrieve job's metadata - this may take a few minutes, depending on the number of jobs stored
    job_ids = [job.jobid for job in Gaia.list_async_jobs()]
    print('job ids',job_ids)
    # Gaia.list_async_jobs() method retrieves a "job" object that contains the job ID of each asynchronous job stored in the user account.
    # Uncomment the following line if you want to delete all the jobs stored in the user space and stop the code there. See below how to delete a subset of the jobs.
    Gaia.remove_jobs(job_ids)

In [2]:
def get_gaia_sample(table="gaiadr3.gold_sample_oba_stars"):

    query = "SELECT * FROM {}".format(table)

    job     = Gaia.launch_job_async(query,verbose=True)
    #job     = Gaia.launch_job(query)
    results = job.get_results()
    print(f'Table size (rows): {len(results)}')
    #results
    return results.to_pandas()

    

In [3]:
def clean_gaia_sample(targets):

    print(targets)

    # select as in paper
    idx = targets['vtan_flag'] == 0

    sel = targets[idx]

    print(len(targets),len(sel))

    return sel

In [7]:
def get_gaia_from_db(ra_min,ra_max,gaiadr='gaiadr3',catname='gaia_source'):
    # Main query, get selected infos with criteria.
    # All data information are given in the Gaia data model: 
    # https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html

    vars = 'source_id, ra, dec, pmra, pmdec, parallax, parallax_error, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, l, b, phot_variable_flag, ref_epoch'
    vars += ', L, B'
    
    if gaiadr == 'gaiadr2':
        vars += ', a_g_val'
    if gaiadr == 'gaiadr3':
        vars += ', ag_gspphot'

    """
    query = "SELECT {} \
    FROM {}.gaia_source \
    WHERE visibility_periods_used > 5 \
    AND astrometric_excess_noise < 0.5 \
    AND parallax > 1 \
    AND parallax_over_error > 5 \
    AND phot_bp_mean_flux_over_error > 20 \
    AND phot_rp_mean_flux_over_error > 20 \
    AND phot_g_mean_flux_over_error > 50 \
    AND phot_bp_rp_excess_factor < 1.2*(1.2+0.03*power(phot_bp_mean_mag-phot_rp_mean_mag,2)) \
    AND ra >= {} \
    and ra < {}".format(vars,gaiadr,ra_min,ra_max)
    """
    query = "SELECT {} \
    FROM {}.{} \
    WHERE visibility_periods_used > 8 \
    AND parallax_over_error > 10 \
    AND phot_bp_mean_flux_over_error > 20 \
    AND phot_rp_mean_flux_over_error > 20 \
    AND phot_g_mean_flux_over_error > 50 \
    AND phot_bp_rp_excess_factor < 1.3+0.06*power(phot_bp_mean_mag-phot_rp_mean_mag,2) \
    AND phot_bp_rp_excess_factor > 1.0+0.015*power(phot_bp_mean_mag-phot_rp_mean_mag,2) \
    AND astrometric_chi2_al/(astrometric_n_good_obs_al-5) < 1.44*greatest(1,exp(-0.4*(phot_g_mean_mag-19.5))) \
    AND ra >= {} \
    and ra < {}".format(vars,gaiadr,catname,ra_min,ra_max)    
    
    print(query)
    job     = Gaia.launch_job_async(query,verbose=True)
    #job     = Gaia.launch_job(query)
    results = job.get_results()
    print(f'Table size (rows): {len(results)}')
    results
    return results.to_pandas()

In [67]:
def get_sourceids(dr='gaiadr3',catalog='astrophysical_parameters'):

    vars = 'source_id'

    query = "SELECT {} FROM {}.{}".format(vars,dr,catalog)

    job     = Gaia.launch_job_async(query,verbose=True)
    #job     = Gaia.launch_job(query)
    results = job.get_results()
    print(f'Table size (rows): {len(results)}')
    results
    return results.to_pandas()

In [8]:
def get_gaia_from_dr(ra_min,ra_max,gaiadr='gaiadr3',catname='gaia_source'):
    
    tt = get_gaia_from_db(ra_min, ra_max,gaiadr,catname)
    ag = 'a_g_val'
    if gaiadr == 'gaiadr3':
        ag = 'ag_gspphot'
    # absolute mag G band - see Gaia Data Release 2 Documentation
    tt['MG'] = tt['phot_g_mean_mag']+5-5*np.log10(1.e3/tt['parallax'])-tt[ag]
    return tt

In [25]:
def get_targets(drs='gaiadr3',catname='gaia_source'):
    ramin = 0
    ramax = 360
    delta_ra = 6
    ras = np.arange(ramin,ramax,delta_ra)
    outDir = '/home/philippe/LSST/gaia_files/{}/{}'.format(drs,catname)
    from sn_tools.sn_io import checkDir
    checkDir(outDir)
    for ra in ras:
        ra_min = np.round(ra,1)
        ra_max = np.round(ra_min+delta_ra,1)
        print(ra_min,ra_max)
        df = get_gaia_from_dr(ra_min,ra_max,drs,catname)
        out_name = '{}/sources_{}_{}.hdf5'.format(outDir,ra_min,ra_max)
        df.to_hdf(out_name,key='star')
        
        

In [33]:
def save_df(rr,drs,catname,key='oba_stars'):
    outDir = '/home/philippe/LSST/gaia_files/{}/{}'.format(drs,catname)
    from sn_tools.sn_io import checkDir
    checkDir(outDir)
    outName = 'sources_{}.hdf5'.format(catname)
    rr.to_hdf('{}/{}'.format(outDir,outName),key=key)
    

In [68]:
Gaia.login(user='pgris', password='Lsst!!2024a=+')

INFO: Login to gaia TAP server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


In [34]:
table="gaiadr3.gold_sample_oba_stars"
tt_spl = table.split('.')
drs = tt_spl[0]
catname = tt_spl[1]
rr = get_gaia_sample(table=table)
save_df(rr,drs,catname,'oba_stars')

Launched query: 'SELECT * FROM gaiadr3.gold_sample_oba_stars'
------>https
host = gea.esac.esa.int:443
context = /tap-server/tap/async
Content-type = application/x-www-form-urlencoded
303 303
[('Date', 'Tue, 21 Oct 2025 06:42:26 GMT'), ('Server', 'Apache/2.4.6 (SLES Expanded Support platform 7) OpenSSL/1.0.2k-fips mod_jk/1.2.43'), ('X-VO-Authenticated', 'pgris'), ('Location', 'https://gea.esac.esa.int/tap-server/tap/async/1761028946091O'), ('Cache-Control', 'no-cache, no-store, max-age=0, must-revalidate'), ('Pragma', 'no-cache'), ('Expires', '0'), ('X-XSS-Protection', '1; mode=block'), ('X-Frame-Options', 'SAMEORIGIN'), ('X-Content-Type-Options', 'nosniff'), ('Transfer-Encoding', 'chunked'), ('Content-Type', 'text/plain;charset=ISO-8859-1')]
job 1761028946091O, at: https://gea.esac.esa.int/tap-server/tap/async/1761028946091O
Retrieving async. results...
INFO: Query finished. [astroquery.utils.tap.core]
Table size (rows): 3023388


In [35]:
table="gaiadr3.gold_sample_fgkm_stars"
tt_spl = table.split('.')
drs = tt_spl[0]
catname = tt_spl[1]
rr = get_gaia_sample(table=table)

Launched query: 'SELECT * FROM gaiadr3.gold_sample_fgkm_stars'
------>https
host = gea.esac.esa.int:443
context = /tap-server/tap/async
Content-type = application/x-www-form-urlencoded
303 303
[('Date', 'Tue, 21 Oct 2025 06:43:24 GMT'), ('Server', 'Apache/2.4.6 (SLES Expanded Support platform 7) OpenSSL/1.0.2k-fips mod_jk/1.2.43'), ('X-VO-Authenticated', 'pgris'), ('Location', 'https://gea.esac.esa.int/tap-server/tap/async/1761029004645O'), ('Cache-Control', 'no-cache, no-store, max-age=0, must-revalidate'), ('Pragma', 'no-cache'), ('Expires', '0'), ('X-XSS-Protection', '1; mode=block'), ('X-Frame-Options', 'SAMEORIGIN'), ('X-Content-Type-Options', 'nosniff'), ('Transfer-Encoding', 'chunked'), ('Content-Type', 'text/plain;charset=ISO-8859-1')]
job 1761029004645O, at: https://gea.esac.esa.int/tap-server/tap/async/1761029004645O
Retrieving async. results...
INFO: Query finished. [astroquery.utils.tap.core]
Table size (rows): 3273041
                   source_id  teff_gspphot  logg_gsppho

TypeError: objects of type ``IntegerArray`` are not supported in this context, sorry; supported objects are: NumPy array, record or scalar; homogeneous list or tuple, integer, float, complex or bytes

In [66]:
tt = rr['evolstage_flame_spec'].unique()
"""
for vv in tt:
    print(vv, type(vv))
"""
rr_cp = pd.DataFrame(rr)
rr_cp["evolstage_flame_spec"] = rr_cp["evolstage_flame_spec"].apply(lambda x:-1 if x is pd.NA else x)
#print(rr_cp['evolstage_flame_spec'].unique())
rr_cp = rr_cp.fillna(int(-999))
rr_cp["spectraltype_esphs"] = rr_cp["spectraltype_esphs"].apply(lambda x:'U' if x=='' else x)
print(rr_cp['spectraltype_esphs'].unique())
rr_cp['spectraltype_esphs'] = rr_cp['spectraltype_esphs'].astype(str)
rr_cp['evolstage_flame'] = rr_cp['evolstage_flame'].astype(int)
print(rr_cp.dtypes)
save_df(rr_cp,drs,catname,'fgkm_stars')

['U' 'F' 'G' 'K' 'M' 'A' 'O' 'B']
source_id                 int64
teff_gspphot            float32
logg_gspphot            float32
mh_gspphot              float32
ag_gspphot              float32
ebpminrp_gspphot        float32
alphafe_gspspec         float32
teff_gspspec            float32
logg_gspspec            float32
mh_gspspec              float32
radius_flame            float32
lum_flame               float32
mass_flame              float32
age_flame               float32
evolstage_flame           int64
radius_flame_spec       float32
lum_flame_spec          float32
mass_flame_spec         float32
age_flame_spec          float32
evolstage_flame_spec    float64
spectraltype_esphs       object
dtype: object


In [70]:
res = get_sourceids()
print(res)

Launched query: 'SELECT source_id FROM gaiadr3.astrophysical_parameters'
------>https
host = gea.esac.esa.int:443
context = /tap-server/tap/async
Content-type = application/x-www-form-urlencoded
303 303
[('Date', 'Tue, 21 Oct 2025 13:30:12 GMT'), ('Server', 'Apache/2.4.6 (SLES Expanded Support platform 7) OpenSSL/1.0.2k-fips mod_jk/1.2.43'), ('X-VO-Authenticated', 'pgris'), ('Location', 'https://gea.esac.esa.int/tap-server/tap/async/1761053412253O'), ('Cache-Control', 'no-cache, no-store, max-age=0, must-revalidate'), ('Pragma', 'no-cache'), ('Expires', '0'), ('X-XSS-Protection', '1; mode=block'), ('X-Frame-Options', 'SAMEORIGIN'), ('X-Content-Type-Options', 'nosniff'), ('Transfer-Encoding', 'chunked'), ('Content-Type', 'text/plain;charset=ISO-8859-1')]
job 1761053412253O, at: https://gea.esac.esa.int/tap-server/tap/async/1761053412253O
Retrieving async. results...


KeyboardInterrupt: 

In [26]:
clean_jobids()
get_targets(drs='gaiadr3',catname='gaia_source')

job ids ['1760972125848O']
INFO: Removed jobs: '['1760972125848O']'. [astroquery.utils.tap.core]
0 6
SELECT source_id, ra, dec, pmra, pmdec, parallax, parallax_error, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, l, b, phot_variable_flag, ref_epoch, L, B, ag_gspphot     FROM gaiadr3.gaia_source     WHERE visibility_periods_used > 8     AND parallax_over_error > 10     AND phot_bp_mean_flux_over_error > 20     AND phot_rp_mean_flux_over_error > 20     AND phot_g_mean_flux_over_error > 50     AND phot_bp_rp_excess_factor < 1.3+0.06*power(phot_bp_mean_mag-phot_rp_mean_mag,2)     AND phot_bp_rp_excess_factor > 1.0+0.015*power(phot_bp_mean_mag-phot_rp_mean_mag,2)     AND astrometric_chi2_al/(astrometric_n_good_obs_al-5) < 1.44*greatest(1,exp(-0.4*(phot_g_mean_mag-19.5)))     AND ra >= 0     and ra < 6
Launched query: 'SELECT source_id, ra, dec, pmra, pmdec, parallax, parallax_error, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, l, b, phot_variable_flag, ref_epoch, L, B, ag_gsp

In [14]:
targets

,source_id,ra,dec,pmra,pmdec,parallax,parallax_error,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,l,b,phot_variable_flag,ref_epoch,L,B,ag_gspphot,MG
12,423289063054369024,0.400430,59.672497,-4.377066,-1.054522,0.457445,0.020055,14.929647,15.329857,14.351916,116.671442,-2.592115,NOT_AVAILABLE,2016.0,116.671442,-2.592115,1.3218,1.909540
36,423289509730936576,0.364285,59.702281,-1.777071,-0.541931,0.333426,0.012067,13.022193,13.334257,12.532269,116.659260,-2.559376,NOT_AVAILABLE,2016.0,116.659260,-2.559376,NaN,NaN
44,423289681529592704,0.345458,59.742632,-3.021024,-2.500900,0.348504,0.014043,14.087346,14.365150,13.647771,116.657728,-2.517951,NOT_AVAILABLE,2016.0,116.657728,-2.517951,1.0354,0.762985
46,423289681529603328,0.377291,59.734461,-2.464894,-1.957056,0.286316,0.012006,12.733460,12.903208,12.425391,116.671907,-2.529062,NOT_AVAILABLE,2016.0,116.671907,-2.529062,1.2705,-1.252813
49,423289715889351808,0.324619,59.714754,-2.017454,-1.542614,0.319811,0.021189,15.105708,15.504957,14.505490,116.642019,-2.543275,NOT_AVAILABLE,2016.0,116.642019,-2.543275,1.8370,0.793179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
896811,1997464292947011328,354.039307,55.493297,2.533718,0.973578,0.408756,0.022502,15.054265,15.326829,14.619435,112.346496,-5.823654,NOT_AVAILABLE,2016.0,112.346496,-5.823654,0.7712,2.340383
896870,1997468175597382016,354.161792,55.596995,-1.126363,-0.618143,0.726699,0.016450,13.568130,13.862082,13.105453,112.443188,-5.744274,NOT_AVAILABLE,2016.0,112.443188,-5.744274,0.8300,2.044904
896938,1997481163578395264,354.102056,55.665601,-2.625141,-2.146707,0.456080,0.020978,15.021451,15.351855,14.516724,112.430525,-5.668873,NOT_AVAILABLE,2016.0,112.430525,-5.668873,1.3510,1.965656
896987,1997485802142965248,354.047289,55.765659,2.938829,-2.523933,1.237782,0.013845,11.613557,11.811242,11.278930,112.429792,-5.564170,NOT_AVAILABLE,2016.0,112.429792,-5.564170,0.5411,1.535677


In [15]:
targets.to_hdf('targets_OBA.hdf5',key='gaiadr3')

In [16]:
sel.to_hdf('OBA_gloden.hdf5',key='gaiadr3')